# 07 - Sync v2 and Phase Continuation Ablation

**Role**: Evaluate the final sync-loss ablation set while keeping `phase_trajectory_sync` as the baseline.

**Comparison set**
- `trajectory_sync` - main sync baseline from notebook 05
- `trajectory_sync_v2` - baseline condition + velocity/SNR sync objective from notebook 05
- `phase_continuation_sync` - phase + phase-advance condition + main sync loss
- `phase_continuation_sync_v2` - phase + phase-advance condition + velocity/SNR sync objective

**Prerequisites**
- Run notebooks 00-04 to build data and train `phase_trajectory_ckpt.pt`.
- Run notebook 05 to produce `frozen_phase_estimator_mlp.pt`, `phase_trajectory_sync_lambda0.12.pt`, and `phase_trajectory_sync_v2_vel0.25_snr10_lambda0.12.pt`.

**Outputs**
- `checkpoints/phase_continuation_ckpt.pt`
- `checkpoints/phase_continuation_sync_lambda0.12.pt`
- `checkpoints/phase_continuation_sync_v2_vel0.25_snr10_lambda0.12.pt`
- `results/table3_sync_phase_continuation_ablation.md`
- `results/sync_phase_continuation_ablation_results.npz`
- `figures/sync_ablation_*.png`


In [ ]:
# Google Drive mount and project-root setup for Colab
from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/phase_conditioned_diffusion_policy')

try:
    from google.colab import drive
except ImportError:
    print(f'Not running in Google Colab; keeping current working directory: {Path.cwd()}')
else:
    drive.mount('/content/drive')
    if not PROJECT_DIR.exists():
        raise FileNotFoundError(
            f'Expected project directory not found: {PROJECT_DIR}\n'
            'Update PROJECT_DIR to the Google Drive folder that contains this repository.'
        )
    os.chdir(PROJECT_DIR)
    print(f'Current working directory: {Path.cwd()}')


In [ ]:
# Colab dependency setup
import importlib.util
import subprocess
import sys

def ensure_package(import_name: str, pip_name: str | None = None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name])

ensure_package('diffusers')
ensure_package('gymnasium')
ensure_package('mujoco')
ensure_package('tabulate')


## 1. Imports and Data

Load artifact paths, the Ant dataset, and shared train/validation loaders. The continuation base model uses the same supervised diffusion objective as trajectory DP, but its per-step condition has four channels: `(cos phi_t, sin phi_t, cos delta_phi_t, sin delta_phi_t)`.


In [ ]:
import gymnasium as gym
import torch

from pcdp.configs import get_experiment_config, set_global_seed
from pcdp.dataset import build_loaders, load_project_data
from pcdp.evaluation import (
    SYNC_CONTINUATION_ABLATION_CONFIG_NAMES,
    SYNC_CONTINUATION_ABLATION_MODEL_KEYS,
    build_frequency_sweep_protocol,
    build_frequency_sweep_results_payload,
    load_evaluation_state,
    print_frequency_sweep_summary,
    run_frequency_sweep_evaluation,
    save_eval_results_npz,
    write_frequency_tracking_table_markdown,
)
from pcdp.experiment_plots import (
    plot_evaluation_frequency_comparison,
    plot_frequency_tracking_alignment,
    plot_zone_aggregated_tracking_metrics,
)
from pcdp.experiment_runner import build_model, build_noise_scheduler, train_or_load_checkpoint
from pcdp.frozen_phase_estimator import (
    freeze_phase_estimator,
    load_phase_estimator_checkpoint,
    train_phase_sync_diffusion_policy,
)
from pcdp.paths import CHECKPOINTS_DIR, DATA_DIR, FIGURES_DIR, RESULTS_DIR, ensure_artifact_dirs
from pcdp.training import load_checkpoint, save_checkpoint, trajectory_phase_continuation_cond_fn

ensure_artifact_dirs()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

data = load_project_data(DATA_DIR)
set_global_seed(data['seed'], deterministic=True)
train_ds, val_ds, train_loader, val_loader = build_loaders(data, batch_size=256, num_workers=2)
print(f"train chunks={len(train_ds):,}, val chunks={len(val_ds):,}")


## 2. Train or Load Phase Continuation Base

This checkpoint is needed because the continuation model changes `per_step_cond_dim` from 2 to 4, so it cannot be initialized from `phase_trajectory_ckpt.pt` without architecture mismatch.


In [ ]:
TRAIN_PHASE_CONTINUATION_BASE = True
OVERWRITE_PHASE_CONTINUATION_BASE = False

base_cfg = get_experiment_config('phase_continuation')
base_ckpt_path = base_cfg.checkpoint_path(CHECKPOINTS_DIR)

base_model = build_model(base_cfg, data, device=device)
base_ema = base_cfg.build_ema(base_model)
base_noise_scheduler, _, _ = build_noise_scheduler(base_cfg)

if base_ckpt_path.exists() and not OVERWRITE_PHASE_CONTINUATION_BASE:
    train_base = False
elif TRAIN_PHASE_CONTINUATION_BASE:
    train_base = True
else:
    raise FileNotFoundError(
        f'Missing phase continuation base checkpoint: {base_ckpt_path}. '
        'Set TRAIN_PHASE_CONTINUATION_BASE=True or copy the checkpoint into CHECKPOINTS_DIR.'
    )

base_train_losses, base_val_log, base_best_ema_state, base_ckpt_path = train_or_load_checkpoint(
    train=train_base,
    cfg=base_cfg,
    model=base_model,
    ema=base_ema,
    noise_scheduler=base_noise_scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    checkpoints_dir=CHECKPOINTS_DIR,
    device=device,
)
print(f'Phase continuation base checkpoint: {base_ckpt_path}')


## 3. Load Frozen Phase Estimator

The estimator remains frozen for all sync fine-tuning runs. Only the diffusion policy is updated.


In [ ]:
estimator_ckpt_path = CHECKPOINTS_DIR / 'frozen_phase_estimator_mlp.pt'
if not estimator_ckpt_path.exists():
    raise FileNotFoundError(
        f'Missing frozen estimator checkpoint: {estimator_ckpt_path}. Run notebook 05 first.'
    )

phase_estimator, estimator_meta = load_phase_estimator_checkpoint(
    estimator_ckpt_path,
    device=device,
    use_best=True,
)
freeze_phase_estimator(phase_estimator)
print('Frozen phase estimator ready.')


## 4. Fine-Tune Phase Continuation Sync Variants

Both variants start from `phase_continuation_ckpt.pt`. The first keeps the main sync objective; the second adds phase-velocity loss and SNR weighting.


In [ ]:
RUN_PHASE_CONTINUATION_SYNC_FINE_TUNE = True
OVERWRITE_PHASE_CONTINUATION_SYNC_CHECKPOINTS = False

SYNC_NUM_EPOCHS = 15
SYNC_LR = 5e-5
LAMBDA_PHASE = 0.12
PHASE_WARMUP_EPOCHS = 1

PHASE_CONTINUATION_SYNC_VARIANTS = [
    {
        'config_name': 'phase_continuation_sync',
        'description': 'phase continuation + main sync loss',
        'phase_abs_weight': 1.0,
        'phase_velocity_weight': 0.0,
        'snr_gamma': None,
        'snr_floor': 0.05,
    },
    {
        'config_name': 'phase_continuation_sync_v2',
        'description': 'phase continuation + mild velocity/weak SNR sync loss',
        'phase_abs_weight': 1.0,
        'phase_velocity_weight': 0.25,
        'snr_gamma': 10.0,
        'snr_floor': 0.05,
    },
]

phase_continuation_sync_logs = {}

for variant in PHASE_CONTINUATION_SYNC_VARIANTS:
    cfg = get_experiment_config(
        variant['config_name'],
        training={
            'num_epochs': SYNC_NUM_EPOCHS,
            'lr': SYNC_LR,
            'val_every': 1,
            'val_n_batches': 8,
            'log_every_step': 100,
        },
    )
    ckpt_path = cfg.checkpoint_path(CHECKPOINTS_DIR)
    print(f"\n=== {cfg.display_name}: {variant['description']} ===")

    if ckpt_path.exists() and not OVERWRITE_PHASE_CONTINUATION_SYNC_CHECKPOINTS:
        print(f'Skipping existing checkpoint: {ckpt_path}')
        continue
    if not RUN_PHASE_CONTINUATION_SYNC_FINE_TUNE:
        raise FileNotFoundError(
            f'Missing checkpoint: {ckpt_path}. Set RUN_PHASE_CONTINUATION_SYNC_FINE_TUNE=True to train it.'
        )

    model = build_model(cfg, data, device=device)
    ema = cfg.build_ema(model)
    noise_scheduler, _, _ = build_noise_scheduler(cfg)

    load_checkpoint(base_ckpt_path, model, ema, device=device, use_best_ema=True)
    train_log, val_log, best_ema_state = train_phase_sync_diffusion_policy(
        model,
        ema,
        noise_scheduler,
        phase_estimator,
        train_loader,
        val_loader,
        cond_fn=trajectory_phase_continuation_cond_fn,
        device=device,
        num_epochs=SYNC_NUM_EPOCHS,
        lr=SYNC_LR,
        weight_decay=cfg.training.weight_decay,
        lambda_phase=LAMBDA_PHASE,
        phase_abs_weight=variant['phase_abs_weight'],
        phase_velocity_weight=variant['phase_velocity_weight'],
        snr_gamma=variant['snr_gamma'],
        snr_floor=variant['snr_floor'],
        phase_warmup_epochs=PHASE_WARMUP_EPOCHS,
        val_every=cfg.training.val_every,
        val_n_batches=cfg.training.val_n_batches,
        grad_clip=cfg.training.grad_clip,
        log_every_step=cfg.training.log_every_step,
    )
    save_checkpoint(
        ckpt_path,
        model,
        ema,
        train_log,
        val_log,
        best_ema_state,
        config={
            **cfg.to_dict(),
            'phase_sync': {
                **variant,
                'lambda_phase': LAMBDA_PHASE,
                'phase_warmup_epochs': PHASE_WARMUP_EPOCHS,
                'source_checkpoint': str(base_ckpt_path),
            },
        },
    )
    phase_continuation_sync_logs[variant['config_name']] = {
        'train_log': train_log,
        'val_log': val_log,
        'checkpoint': ckpt_path,
    }


## 5. Frequency-Sweep Ablation Evaluation

This compares the main sync baseline against the three ablations under the same target-frequency grid. The result is intentionally separate from notebook 06 so the final main table stays focused.


In [ ]:
RUN_ABLATION_EVALUATION = True
N_SEEDS_SWEEP = 10
MAX_STEPS = 1000
DT = 0.05

if RUN_ABLATION_EVALUATION:
    freq_protocol = build_frequency_sweep_protocol(data, n_in_dist=3, ood_iqr_scale=1.5)
    state = load_evaluation_state(
        data,
        device=device,
        checkpoints_dir=CHECKPOINTS_DIR,
        config_names=SYNC_CONTINUATION_ABLATION_CONFIG_NAMES,
    )

    env = gym.make('Ant-v5')
    try:
        ablation_freq_results = run_frequency_sweep_evaluation(
            state,
            freq_protocol,
            env=env,
            data=data,
            device=device,
            model_keys=SYNC_CONTINUATION_ABLATION_MODEL_KEYS,
            n_seeds=N_SEEDS_SWEEP,
            max_steps=MAX_STEPS,
            dt=DT,
        )
    finally:
        env.close()

    print_frequency_sweep_summary(
        data,
        freq_protocol,
        ablation_freq_results,
        model_keys=SYNC_CONTINUATION_ABLATION_MODEL_KEYS,
    )

    table_path = write_frequency_tracking_table_markdown(
        freq_protocol,
        ablation_freq_results,
        RESULTS_DIR / 'table3_sync_phase_continuation_ablation.md',
        model_keys=SYNC_CONTINUATION_ABLATION_MODEL_KEYS,
        interval='ci95',
    )

    payload = build_frequency_sweep_results_payload(
        data,
        freq_protocol,
        ablation_freq_results,
        model_keys=SYNC_CONTINUATION_ABLATION_MODEL_KEYS,
        n_seeds_sweep=N_SEEDS_SWEEP,
    )
    npz_path = save_eval_results_npz(
        payload,
        RESULTS_DIR / 'sync_phase_continuation_ablation_results.npz',
    )
    print(table_path)
    print(npz_path)
else:
    print('Set RUN_ABLATION_EVALUATION=True to run rollouts.')


## 6. Ablation Figures

These figures use the same plotting style as the main evaluation but only for the ablation comparison set.


In [ ]:
if RUN_ABLATION_EVALUATION:
    plot_evaluation_frequency_comparison(
        {},
        ablation_freq_results,
        data,
        FIGURES_DIR / 'sync_ablation_reward_per_step_vs_freq.png',
        n_seeds_sweep=N_SEEDS_SWEEP,
        phase_model_keys=SYNC_CONTINUATION_ABLATION_MODEL_KEYS,
        state=state,
    )
    plot_frequency_tracking_alignment(
        ablation_freq_results,
        data,
        FIGURES_DIR / 'sync_ablation_target_vs_measured_freq.png',
        n_seeds_sweep=N_SEEDS_SWEEP,
        phase_model_keys=SYNC_CONTINUATION_ABLATION_MODEL_KEYS,
        state=state,
        interval='ci95',
        periodic_collapse_note=False,
    )
    plot_zone_aggregated_tracking_metrics(
        freq_protocol,
        ablation_freq_results,
        FIGURES_DIR / 'sync_ablation_zone_tracking_metrics.png',
        phase_model_keys=SYNC_CONTINUATION_ABLATION_MODEL_KEYS,
        state=state,
        interval='ci95',
    )
